# Aprendizaje Automático
# Trabajo Práctico 2

Profesor: Juan Luis Crespo Mariño

Instituto Tecnológico de Costa Rica,

Programa Ciencia de Datos

---

Fecha de entrega: 11 de agosto, hora límite las 6:00 pm.

Medio de entrega: Por medio del TEC-Digital.

Entregables: Un archivo jupyter ( .IPYNB ).

BD utilizada: https://archive.ics.uci.edu/dataset/2/adult


Estudiante:
1. **Jose Pablo Ruiz Myrie**
2. **Nikole Villalobos Lopez**


# Notebook 03 — Modelo de Regresión Logística

En este notebook se entrena y evalúa una Regresión Logística como modelo
baseline para clasificar si el ingreso anual de una persona es superior a $50K.
Se utiliza la selección de variables y las decisiones de preprocesado definidas
en los notebooks anteriores.


In [ ]:
pip install ucimlrepo

In [6]:
# ============================================================
# PREPARACIÓN DEL DATASET
# ============================================================

import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo
from sklearn.impute import SimpleImputer

#Cargamos el dataset
df = pd.read_csv("../data/datos_procesados.csv")

print("Dataset preparado:", df.shape)
df.head()


Dataset preparado: (48842, 17)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,capital-gain-log,capital-loss-log
0,22,Private,174043,HS-grad,9,Never-married,Craft-repair,Not-in-family,White,Male,0,0,50,United-States,<=50K,0.000000,0.0
1,24,Private,399449,Bachelors,13,Never-married,Sales,Own-child,White,Female,0,0,40,United-States,<=50K,0.000000,0.0
2,44,Self-emp-inc,79521,Bachelors,13,Married-civ-spouse,Farming-fishing,Husband,White,Male,15024,0,55,United-States,>50K.,9.617471,0.0
3,25,Private,352806,HS-grad,9,Divorced,Other-service,Not-in-family,White,Female,0,0,40,Mexico,<=50K,0.000000,0.0
4,56,Self-emp-not-inc,52822,Some-college,10,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,70,United-States,<=50K.,0.000000,0.0


In [ ]:
# ============================================================
#  Selección de algoritmos y partición de datos
# ============================================================

# Se selecciona Regresión logística debido a que el problema que deseamos
# resolver es determinar si una persona gana más de 50k $ o no. Es decir,
# la variable objetivo es cátegorica binaria. Por lo que tenemos una tarea
# de clasificación. El modelo estima la probabilidad de pertenecer a una de las
# clases. La regresión logística pertenece a la familia de modelos lineales.
# Entre sus ventajas están: rapidez, interpretabilidad y que sus
# coeficientes permiten analizar la dirección e importancia relativa de las
# variables

# Se utiliza una partición 80 % entrenamiento / 20 % prueba donde la mayor parte
# de los datos se utiliza para que el modelo aprenda.

In [ ]:
# ============================================================
# Entrenamiento y ajuste de hiperparámetros
# ============================================================
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Preparar variable objetivo
# ------------------------------------------------------------
df_model = df.copy()

# Unificar etiquetas de income
df_model["income"] = (
    df_model["income"]
    .str.strip()
    .str.rstrip(".")
)

# Convertir la variable objetivo a 0 y 1 donde  0 = <=50K
# y 1 = >50K

df_model["income"] = df_model["income"].map({
    "<=50K": 0,
    ">50K": 1
})

# ------------------------------------------------------------
# 2. Selección de variables
# ------------------------------------------------------------

selected_features_model = [
    "age",
    "workclass",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain-log",
    "capital-loss-log",
    "hours-per-week",
    "native-country"
]

X_model = df_model[selected_features_model]
y_model = df_model["income"]

# ------------------------------------------------------------
# 3.Separamiento de entrenamiento y prueba
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model
)

# ------------------------------------------------------------
# 4.Ajustamos preprocesado únicamente al conjunto de entrenamiento
# ------------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

# Variables numéricas que utilizan StandardScaler
standard_columns = [
    "age",
    "education-num",
    "capital-gain-log",
    "capital-loss-log"
]

# Variables numéricas que utilizan RobustScaler
robust_columns = [
    "hours-per-week"
]

# Variables categóricas
categorical_columns = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

#Construimos el preprocesador

preprocessor_model = ColumnTransformer(
    transformers=[
        (
            "standard",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            standard_columns
        ),

        (
            "robust",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler())
            ]),
            robust_columns
        ),

        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "onehot",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore"
                    )
                )
            ]),
            categorical_columns
        )
    ]
)

# ------------------------------------------------------------
# 5. Regresión logistica baseline
# ------------------------------------------------------------
from sklearn.linear_model import LogisticRegression

logistic_baseline = Pipeline([
    (
        "preprocessor",
        preprocessor_model
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

# Entrenamiento
logistic_baseline.fit(
    X_train,
    y_train
)

# Predicciones
y_pred_baseline = logistic_baseline.predict(X_test)

y_prob_baseline = logistic_baseline.predict_proba(X_test)[:, 1]


In [ ]:
# ============================================================
# Evaluación comparativa
# ============================================================

# Realizamos la evaluación de la Regresión Logística mediante las métricas
# Accuracy, Precision, Recall, F1 y AUC-ROC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

baseline_metrics = {
    "Modelo": "Regresión Logística - Baseline",

    "Accuracy": accuracy_score(
        y_test,
        y_pred_baseline
    ),

    "Precision": precision_score(
        y_test,
        y_pred_baseline
    ),

    "Recall": recall_score(
        y_test,
        y_pred_baseline
    ),

    "F1": f1_score(
        y_test,
        y_pred_baseline
    ),

    "AUC-ROC": roc_auc_score(
        y_test,
        y_prob_baseline
    )
}

for metric, value in baseline_metrics.items():
    if metric != "Modelo":
        print(f"{metric}: {value:.4f}")



Accuracy: 0.8431
Precision: 0.7089
Recall: 0.5843
F1: 0.6406
AUC-ROC: 0.8958


In [ ]:
# ============================================================
# Interpretación y análisis de variables
# ============================================================